[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-orientation.ipynb)

# Rationale 1 Orientation: Detecting Plant Stress

##### Welcome. This notebook gets you ready to work on Rationale 1 of the Virtual Lab, which is about using machine learning to study how plants respond to stress at the level of gene expression. By the end of this notebook you will have:

- **A working environment in Google Colab, with Google Drive connected so your work is saved.**
- **The `irilab2026` package installed, which contains the tools you'll use across all four R1 notebooks.**
- **The AtGenExpress dataset loaded — a set of microarray experiments measuring how *Arabidopsis thaliana* responds to nine different kinds of stress.**
- **A first look at what the data looks like, so the analytical notebooks that come next start from familiar ground.**

You don't need to install anything on your computer. Everything runs in Colab. To begin, click the cells in order from top to bottom and run each one. We'll explain what each cell does as we go.

In [ ]:
# First we install the base package, using the package manager `pip`
!pip install git+https://github.com/geraldmc/irilab2026.git -q # for actual use we need a tag specified.

# What just happened? pip pulled the irilab2026 package from GitHub and installed it.
# Now we can import the package and use it in our code.
import irilab2026

#### Before we use the one-line setup, let's see what it actually does. There are three parts.

In [ ]:
from irilab2026 import is_colab, mount_google_drive, has_gpu

# Part 1. Detect Colab.
is_colab()
# Part 2. Mount Google Drive.
mount_google_drive()
# Part 3. Check for a GPU.
has_gpu()

##### Now that we've seen the three steps, here's a one-line version that ALL notebooks will use. Once this runs you don't need to run it again — Google Drive is already mounted.

In [ ]:
from irilab2026 import setup

# This is the first cell of every analytical notebook in this program.
# We don't need to run it here because we already did the three pieces above.
# setup(gpu_required=False)

### What is AtGenExpress? 

#### Before we load the data, it's worth knowing what we're loading.
##### **What is a microarray?**
A **microarray** is a piece of glass — about the size of a postage stamp — with thousands of tiny spots arranged in a grid. Each spot contains short pieces of DNA whose sequence matches one specific gene in the Arabidopsis genome. The microarray we'll be working with, called ATH1, has about 22,000 such spots, covering most of the genes in the plant.

To use a microarray, you take a biological sample — for example, leaves from a plant that has been exposed to cold for 24 hours — and extract the messenger RNA (mRNA) from it. Messenger RNA is what cells make when they want to use a gene; the more a gene is being used, the more mRNA there is. You label the mRNA with a fluorescent tag, wash it over the microarray, and the molecules stick to the spots whose DNA they match. Then you scan the array and measure how brightly each spot glows. A bright spot means lots of mRNA stuck there, which means that gene was being heavily used in the sample. A dim spot means the opposite.

The number you get for each spot is called the expression value for that gene. We'll come back to that in a moment, after we load the data.

In [ ]:
from irilab2026 import load_atgenexpress

data = load_atgenexpress()

# The data structure the function returns `data` is a Python dictionary. 
# Each key is a stress name, each value is a table of expression measurements.

data.keys()

### Three caveats to carry forward

AtGenExpress is a clean dataset, but every dataset has decisions baked into its design that affect how its results should be read. Three of those decisions matter enough that you should know about them before looking at any of the data.

##### Shoot and root samples are not equivalent across stresses

For each stress, AtGenExpress measured both shoot tissue (above-ground) and root tissue (below-ground). It's tempting to read this as "every stress has a shoot response and a root response, and we can compare them." But the comparison isn't symmetric across stresses. Five of the eight stresses were applied locally to one tissue — for example, salt was applied to the roots, UV-B to the shoots — so the other tissue is showing a systemic response, not a direct one. The remaining three stresses (cold, drought, heat) were applied to the whole plant, so both tissues are directly stressed. This means "shoot vs. root" means different things in different stresses, and any analysis that pools them or compares them needs to be designed with that in mind.

##### Drought in AtGenExpress is not the same as drought in nature

The AtGenExpress drought condition was produced by lifting plants out of their hydroponic growth solution and letting them dry on a rack — losing about 10% of their fresh weight in a matter of hours. That captures a real, measurable plant response, but it's not the same biology as field drought, where soil dries out over days or weeks and the plant adjusts gradually. A gene that lights up in AtGenExpress drought may or may not be relevant to field drought; the two conditions engage overlapping but distinct response programs. If you find something interesting in the drought data, treat it as "what Arabidopsis does under fast desiccation," not "what Arabidopsis does under drought in general."

##### Eight stresses, not nine

The full AtGenExpress series was designed with nine stresses, but the ninth — oxidative stress — was only ever deposited in TAIR/NASCArrays, not in GEO. Because our loader pulls from GEO, you'll see eight stresses in the data, not nine. This isn't a bug in the loader; it's a deliberate choice that matches what's actually available in the public GEO download path. If you ever need oxidative-stress data, you'll have to get it from a different source.

### First Look at The Data

We've loaded the data, but right now it's just a Python variable. Let's actually look at it.We're going to look at three things: how big the dataset is, what it looks like up close, and what the values themselves are like. Each one will probably teach you something — about microarray data in general, and about what kinds of analyses this dataset will and won't support.

#### How big is this dataset?

Most data tables you've worked with are taller than they are wide — a spreadsheet of grades has a row per student and maybe ten columns, a customer database has a row per customer and a few dozen columns. Microarray data is the opposite. It's very wide (many measurements) and very short (few samples). Let's see how wide and how short.

In [ ]:
for stress_name, df in data.items():
    print(f"{stress_name:12s} {df.shape[0]:>6d} probes × {df.shape[1]:>3d} samples")

About **22,810 probes** for every stress — that's the size of the ATH1 microarray, fixed across the whole experiment. But only a few dozen samples per stress. That ratio — many features, few observations — is the foundational fact about microarray data, and it shapes everything we'll do with it. Many statistical methods that work fine on tall-and-narrow data behave differently when there are far more measurements than samples; we'll come back to this in R1-Q1.

#### What does the data actually look like?

Now let's zoom in. Here's the top-left corner of the cold-stress table — the first 5 probes and the first 5 samples.

In [ ]:
data['cold'].iloc[:5, :5]

#### Three things to notice:

The row labels (like `AFFX-r2-P1-cre-5_at`) are probe IDs — identifiers for the spots on the microarray. They're not gene names yet. To get from a probe ID to a gene we'd recognize (say, `AT1G01010`), we need an annotation table that maps one to the other. We'll handle that in R1-Q1's first analytical notebook.

The column labels (like `GSM131223`) are sample IDs assigned by GEO, the database we downloaded the data from. The IDs themselves are uninformative, but each one corresponds to a specific tissue (shoot or root), time point, and replicate. That information is encoded in the GEO sample titles, which we haven't parsed yet — also a job for R1-Q1.

The values in the cells are expression levels. Each one is the measurement of how strongly that probe lit up in that sample. They're positive numbers spanning a wide range — some in the hundreds, some in the thousands. That's normal; we'll see why in the next cell.

##### What do the values look like across all probes?

The 5×5 peek showed us a few values, but we have hundreds of thousands of them. To get a sense of the whole dataset, we'll look at the distribution of values — how often each value appears. We'll use cold-stress as our example.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

values = data['cold'].values.flatten()

plt.figure(figsize=(8, 4))

sns.histplot(
    x=values,
    bins=100,
    edgecolor='white'
)

plt.xlabel('Expression value')
plt.ylabel('Number of probes × samples')
plt.title('Distribution of expression values: cold stress')
plt.yscale('log')

sns.despine()
plt.tight_layout()
plt.show()

Two things about this distribution. First, it's heavily right-skewed — the bulk of values are small (most probes have low expression most of the time), and a long tail stretches out toward high values (a smaller number of probes are highly expressed). This is normal for gene expression data; cells don't use most genes very much, and use a few genes a lot.

Second, the y-axis is on a log scale. If we plotted it on a regular linear scale, the tall bars near zero would be so much taller than everything else that the long tail would look flat. The log scale compresses the y-axis so we can see both the bulk and the tail at once. This is a common move when looking at distributions like this — keep an eye out for it in the analytical notebooks.